In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Estimator によるノイズ管理の設定

<Accordion>
<AccordionItem title="パッケージ・バージョン">

このページのコードは、以下の要件を使用して開発されました。
これらのバージョン以降の使用をお勧めします。

```
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>

ノイズを管理する方法はいくつかあります。一般的に、エラーが発生する前に回避するためのさまざまなエラー緩和およびエラー抑制技術を使用します。これらの技術は通常、前処理のオーバーヘッドを引き起こします。そのため、結果を完璧にすることと、ジョブが適切な時間内に完了することのバランスを取ることが重要です。

Estimator は以下のノイズ管理技術をサポートしています。各技術の説明については、[エラー緩和と抑制技術](error-mitigation-and-suppression-techniques)を参照してください。これらの技術を有効にする手順については、[カスタム・エラー設定](#advanced-error)セクションを参照してください。

- [ダイナミカル・デカップリング](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-dynamical-decoupling-options#dynamicaldecouplingoptions)
- [Pauli ツワーリング](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)
- [PEA](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options)
- [PEC](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#pec)
- [`resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)
- [TREX](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#measure_mitigation)
- [ZNE](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#zne)

<span id="resilience"></span>
## レジリエンス・レベル
`resilience_level` は、エラーに対してどの程度の耐性を構築するかを指定します。レベルが高いほど、処理時間が長くなる代わりに、より正確な結果が得られます。レジリエンス・レベルを使用して、プリミティブ・クエリーにノイズ管理を適用する際のコスト・精度のトレードオフを設定できます。ノイズ管理は、関連する回路のコレクション（アンサンブル）の出力を処理することで、結果のエラー（バイアス）を低減します。エラー低減の程度は、適用する手法によって異なります。レジリエンス・レベルは、ノイズ管理手法の詳細な選択を抽象化して、ユーザーがアプリケーションに適したコスト・精度のトレードオフについて考えられるようにします。

そのため、各レベルは、さまざまな時間・精度のトレードオフを試せるように、量子サンプリングのオーバーヘッドが増加する手法または手法群に対応しています。次の表は、各プリミティブで使用可能なレベルと対応する手法を示しています。

<span id="resilience-table"></span>

| レジリエンス・レベル | 説明 | 技術 |
|------------------|-------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------|
| 0 | 緩和なし | なし |
| 1 [デフォルト] | 最小限の緩和コスト：読み出しエラーに関連するエラーを緩和 | [ツワーリング読み出しエラー消去（TREX）](/guides/error-mitigation-and-suppression-techniques#trex)測定ツワーリング |
| 2 | 中程度の緩和コスト。通常、推定量のバイアスを低減しますが、ゼロバイアスであることは保証されません。 | レベル 1 + [ゼロノイズ外挿（ZNE）](/guides/error-mitigation-and-suppression-techniques#zne)とゲート・ツワーリング |

> **Info:** レジリエンス・レベルは現在ベータ版であるため、サンプリング・オーバーヘッドとソリューション品質は回路によって異なります。新機能、高度なオプション、管理ツールは段階的にリリースされます。特定のノイズ管理手法が各レジリエンス・レベルで適用されることは保証されていません。

### レジリエンス・レベルを使用した Estimator の設定
レジリエンス・レベルを使用してノイズ管理技術を指定するか、[カスタム・エラー設定](#advanced-error)で説明されているように技術を個別にカスタム設定することができます。

> **Note:** レジリエンス・レベルに加えて手動で指定したオプションは、レジリエンス・レベルで定義された基本オプション・セットに追加して適用されます。したがって、原則的に、レジリエンス・レベルを 1 に設定しても測定緩和をオフにすることができますが、これは推奨されません。
> 
> 例えば、レジリエンス・レベルを 0 に設定すると `zne_mitigation` がオフになりますが、`estimator.options.resilience.zne_mitigation = True` はその値をオーバーライドします。

### 例
以下のコードは、`resilience_level 2` を設定することで ZNE、TREX、ゲート・ツワーリングを有効にします。

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

<span id="advanced-error"></span>
## カスタム・ノイズ管理設定
[Estimator オプション](/guides/estimator-options)を使用して、個別のノイズ管理手法のオン・オフを切り替えることができます。

> **Note:** すべてのオプションがすべての種類の回路で連携して機能するわけではありません。詳細については、[機能互換性テーブル](estimator-options#options-compatibility-table)を参照してください。

### 例

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(backend)
options = estimator.options
# Turn on gate twirling.
options.twirling.enable_gates = True
# Turn on measurement error mitigation.
options.resilience.measure_mitigation = True

print(
    f">>> gate twirling is turned on: {estimator.options.twirling.enable_gates}"
)
print(
    f">>> measurement error mitigation is turned on: {estimator.options.resilience.measure_mitigation}"
)

>>> gate twirling is turned on: True
>>> measurement error mitigation is turned on: True


## Turn off all error mitigation

For instructions to turn off all error mitigation see the [Turn off all error suppression and mitigation](estimator-options#no-error-mitigation) section in the Estimator options guide.

## Next steps

<Admonition type="tip" title="Recommendations">
  - Walk through an example that uses error mitigation in the [Cost function lesson](/learning/courses/variational-algorithm-design/cost-functions) in IBM Quantum Learning.
  - Learn more about [error mitigation and error suppression techniques](error-mitigation-and-suppression-techniques).
  - Explore [Estimator options](/docs/guides/estimator-options).
  - Decide what [execution mode](/docs/guides/execution-modes) to run your job in.
</Admonition>